# Poisson Regression: Predicting Goals in Matches\n\n**Chapter 6: Regression Techniques for Soccer Analytics**\n\n## Learning Objectives\n\n- Understand why Poisson regression is ideal for count data\n- Learn the difference between linear and Poisson regression\n- Build a Poisson model to predict goals scored\n- Use statsmodels for GLM (Generalized Linear Models)\n- Interpret coefficients on a logarithmic scale\n- Visualize non-linear relationships

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport statsmodels.api as sm\nimport statsmodels.formula.api as smf\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (12, 8)

## Why Poisson Regression for Goals?\n\nLinear regression is great for continuous values, but **goals are counts**. Here's why Poisson is better:\n\n| Issue | Linear Regression | Poisson Regression |\n|-------|-------------------|-------------------|\n| **Fractional values** | Can predict 2.7 goals | Only predicts whole numbers |\n| **Negative values** | Can predict -1 goals | Only predicts non-negative |\n| **Relationship** | Assumes linear | Handles non-linear patterns |\n\n**Poisson regression** is a type of **Generalized Linear Model (GLM)** specifically designed for count data.

## The Poisson Distribution\n\n**Key assumptions:**\n1. Events (goals) occur at a constant average rate\n2. Events are independent\n3. Two events cannot occur at exactly the same instant\n\nThis fits soccer matches well! Goals are discrete events that occur independently throughout a match.

## Load Data: Goals Scored\n\nWe'll predict the number of goals a team scores based on shots on target.

In [ ]:
# Simulated data for goals scored\ndata = {\n    'Team': [f'Team {i}' for i in range(1, 21)],\n    'ShotsOnTarget': [5, 8, 12, 4, 6, 9, 10, 3, 7, 11, 5, 8, 12, 4, 6, 9, 10, 3, 7, 11],\n    'Goals': [1, 2, 3, 0, 1, 2, 2, 0, 1, 3, 1, 2, 4, 1, 1, 2, 3, 0, 2, 3]\n}\ngoals_df = pd.DataFrame(data)\n\nprint(\"Goals Dataset:\")\nprint(goals_df.head(10))\nprint(f\"\\nDataset shape: {goals_df.shape}\")\nprint(f\"\\nGoals distribution:\")\nprint(goals_df['Goals'].value_counts().sort_index())

## Visualize the Data

In [ ]:
plt.figure(figsize=(10, 6))\nsns.scatterplot(x='ShotsOnTarget', y='Goals', data=goals_df, s=100)\nplt.title('Goals vs. Shots on Target', fontsize=14)\nplt.xlabel('Shots on Target')\nplt.ylabel('Number of Goals')\nplt.tight_layout()\nplt.show()\n\nprint(\"Notice: Goals are discrete (0, 1, 2, 3...), not continuous.\")\nprint(\"This is why we need Poisson regression!\")

## Build the Poisson Regression Model\n\nWe'll use **statsmodels** which provides excellent support for GLMs and detailed statistical summaries.

In [ ]:
# Build the Poisson model using formula notation\nformula = \"Goals ~ ShotsOnTarget\"\nmodel = smf.poisson(formula, data=goals_df).fit()\n\n# Print model summary\nprint(model.summary())

## Interpret the Coefficients\n\n**Important:** Poisson regression uses a **log link function**, so coefficients are on a logarithmic scale.\n\nTo interpret:\n- **Exponentiate the coefficient**: `exp(coef)` gives the multiplicative effect\n- If `coef = 0.15`, then `exp(0.15) ≈ 1.16`\n- **Interpretation:** Each additional shot on target multiplies expected goals by 1.16 (a 16% increase)

In [ ]:
# Extract and interpret coefficients\ncoef_shots = model.params['ShotsOnTarget']\nmultiplicative_effect = np.exp(coef_shots)\n\nprint(f\"Coefficient for ShotsOnTarget: {coef_shots:.4f}\")\nprint(f\"Multiplicative effect: {multiplicative_effect:.4f}\")\nprint(f\"\\nInterpretation:\")\nprint(f\"Each additional shot on target increases expected goals by {(multiplicative_effect-1)*100:.1f}%\")

## Visualize the Poisson Model\n\nUnlike linear regression, Poisson creates a **curved prediction line**.

In [ ]:
# Get model predictions\ngoals_df['predicted_goals'] = model.predict(goals_df)\n\nplt.figure(figsize=(10, 6))\n\n# Plot actual goals\nsns.scatterplot(x='ShotsOnTarget', y='Goals', data=goals_df, label='Actual Goals', s=100)\n\n# Plot predicted goals (curved line)\nsorted_df = goals_df.sort_values('ShotsOnTarget')\nplt.plot(sorted_df['ShotsOnTarget'], sorted_df['predicted_goals'], \n         color='red', linewidth=3, linestyle='--', label='Predicted Goals (Poisson)')\n\nplt.title('Poisson Regression: Goals vs. Shots on Target', fontsize=14)\nplt.xlabel('Shots on Target')\nplt.ylabel('Number of Goals')\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint(\"Notice: The prediction line curves upward!\")\nprint(\"This captures the non-linear relationship better than a straight line.\")

## Make Predictions\n\nPredict goals for teams with different shot counts.

In [ ]:
# Create new data for prediction\nnew_teams = pd.DataFrame({\n    'ShotsOnTarget': [3, 6, 9, 12, 15]\n})\n\n# Predict expected goals\npredictions = model.predict(new_teams)\n\nprint(\"Expected Goals Predictions:\")\nfor shots, goals in zip(new_teams['ShotsOnTarget'], predictions):\n    print(f\"  {shots} shots on target → {goals:.2f} expected goals\")

## Compare: Linear vs. Poisson\n\nLet's see the difference between linear and Poisson regression.

In [ ]:
from sklearn.linear_model import LinearRegression\n\n# Build linear model\nX = goals_df[['ShotsOnTarget']]\ny = goals_df['Goals']\nlinear_model = LinearRegression()\nlinear_model.fit(X, y)\ngoals_df['linear_pred'] = linear_model.predict(X)\n\n# Plot comparison\nplt.figure(figsize=(12, 6))\nsns.scatterplot(x='ShotsOnTarget', y='Goals', data=goals_df, s=100, label='Actual')\n\nsorted_df = goals_df.sort_values('ShotsOnTarget')\nplt.plot(sorted_df['ShotsOnTarget'], sorted_df['predicted_goals'], \n         color='red', linewidth=3, label='Poisson', linestyle='--')\nplt.plot(sorted_df['ShotsOnTarget'], sorted_df['linear_pred'], \n         color='blue', linewidth=3, label='Linear', linestyle='-.')\n\nplt.title('Poisson vs. Linear Regression', fontsize=14)\nplt.xlabel('Shots on Target')\nplt.ylabel('Goals')\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint(\"Key Difference:\")\nprint(\"- Linear: Straight line (can predict negative/fractional goals)\")\nprint(\"- Poisson: Curved line (always non-negative, better for counts)\")

## Model Evaluation\n\nFor count data, we can use metrics like **Mean Absolute Error (MAE)**.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error\n\n# Calculate errors\nmae_poisson = mean_absolute_error(goals_df['Goals'], goals_df['predicted_goals'])\nmae_linear = mean_absolute_error(goals_df['Goals'], goals_df['linear_pred'])\n\nprint(\"Model Performance Comparison:\")\nprint(f\"  Poisson MAE: {mae_poisson:.3f}\")\nprint(f\"  Linear MAE:  {mae_linear:.3f}\")\n\nif mae_poisson < mae_linear:\n    print(\"\\nPoisson regression performs better for this count data!\")\nelse:\n    print(\"\\nLinear regression performs similarly (but Poisson is still more appropriate for counts)\")

## Summary\n\nIn this notebook, we:\n\n1. ✅ Learned why Poisson regression is ideal for count data\n2. ✅ Built a Poisson model to predict goals from shots on target\n3. ✅ Used statsmodels for GLM modeling\n4. ✅ Interpreted coefficients on a logarithmic scale\n5. ✅ Visualized the non-linear relationship\n6. ✅ Compared Poisson vs. linear regression\n\n## Key Takeaways\n\n- **Poisson regression** is designed for count data (goals, cards, corners)\n- Uses a **log link function** → coefficients need exponentiation\n- Produces **curved predictions** that respect count data properties\n- Always **non-negative** predictions\n- Better than linear regression for discrete outcomes\n\n## Next Steps\n\nIn the next notebook, we'll explore **K-Nearest Neighbors (KNN) Regression** for finding similar players!

## Practice Exercises\n\n1. **Multiple Predictors**: Add features like possession, xG, or opponent strength\n2. **Match Outcome Prediction**: Predict goals for both teams and determine winner\n3. **Poisson Distribution**: Plot the Poisson distribution for different lambda values\n4. **Real Match Data**: Apply to actual match data from StatsBomb or similar\n5. **Overdispersion Check**: Research and test if the data shows overdispersion